In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clean_orders.csv',
                 parse_dates=['order_date', 'ship_date', 'order_month'])
print(df.shape)

(172765, 20)


In [2]:
df.groupby('order_month')['order_id'].nunique()

order_month
2015-01-01    1713
2015-02-01    1505
2015-03-01    1702
2015-04-01    1639
2015-05-01    1684
2015-06-01    1651
2015-07-01    1704
2015-08-01    1678
2015-09-01    1634
2015-10-01    1696
2015-11-01    1658
2015-12-01    1731
2016-01-01    1703
2016-02-01    1574
2016-03-01    1700
2016-04-01    1630
2016-05-01    1675
2016-06-01    1617
2016-07-01    1662
2016-08-01    1698
2016-09-01    1664
2016-10-01    1703
2016-11-01    1639
2016-12-01    1690
2017-01-01    1685
2017-02-01    1538
2017-03-01    1710
2017-04-01    1666
2017-05-01    1672
2017-06-01    1605
2017-07-01    1693
2017-08-01    1695
2017-09-01    1640
2017-10-01    2019
2017-11-01    1956
2017-12-01    2031
2018-01-01    2037
Name: order_id, dtype: int64

In [3]:
start, end = '2015-01-01', '2017-09-01'
d = df[(df['order_month'] >= start) & (df['order_month'] <= end)]
months = pd.date_range(start, end, freq='MS')

rev = (d.groupby(['product_id', 'product'])['sales'].sum()
         .sort_values(ascending=False).reset_index())
rev['share'] = rev['sales'] / rev['sales'].sum()
prior = rev['share'].cumsum() - rev['share']
rev['ABC'] = np.where(prior < 0.80, 'A', np.where(prior < 0.95, 'B', 'C'))

demand = (d.groupby(['product_id', 'order_month'])['qty'].sum()
            .unstack(fill_value=0).reindex(columns=months, fill_value=0))
stats = pd.DataFrame({
    'avg_monthly_demand': demand.mean(axis=1),
    'std_demand': demand.std(axis=1),
    'active_months': (demand > 0).sum(axis=1)
})
stats['cv'] = stats['std_demand'] / stats['avg_monthly_demand']
stats['XYZ'] = np.where(stats['cv'] <= 0.5, 'X', np.where(stats['cv'] <= 1.0, 'Y', 'Z'))

seg = rev.merge(stats, left_on='product_id', right_index=True, how='left')
seg['segment'] = seg['ABC'] + seg['XYZ']

print(len(set(df['product_id']) - set(d['product_id'])), "products have no sales in the window")
print(pd.crosstab(seg['ABC'], seg['XYZ'], margins=True))
seg.to_csv('../data/product_segments.csv', index=False)

18 products have no sales in the window
XYZ  X   Y   Z  All
ABC                
A    7   0   0    7
B    2   3   0    5
C    0  42  46   88
All  9  45  46  100


In [4]:
life = df.groupby('product_id')['order_month'].agg(first_month='min', last_month='max')
cut = pd.Timestamp('2017-10-01')
life['status'] = np.where(life['first_month'] >= cut, 'New (Oct 2017+)',
                  np.where(life['last_month'] < cut, 'Discontinued', 'Active'))
print(life['status'].value_counts(), '\n')

seg2 = seg.merge(life[['status']], left_on='product_id', right_index=True, how='left')
summary = seg2.groupby('status').agg(products=('product_id', 'count'), revenue=('sales', 'sum'))
summary['revenue_%'] = (summary['revenue'] / summary['revenue'].sum() * 100).round(1)
print(summary, '\n')
print(stats.groupby('XYZ')['active_months'].describe().round(1))

status
Discontinued       87
New (Oct 2017+)    18
Active             13
Name: count, dtype: int64 

              products       revenue  revenue_%
status                                         
Active              13  3.102902e+07       94.6
Discontinued        87  1.759228e+06        5.4 

     count  mean  std   min   25%   50%   75%   max
XYZ                                                
X      9.0  33.0  0.0  33.0  33.0  33.0  33.0  33.0
Y     45.0  28.0  0.0  28.0  28.0  28.0  28.0  28.0
Z     46.0   4.5  2.3   1.0   1.0   6.0   6.0   6.0


In [5]:
dcat = (d.groupby(['category', 'order_month'])['qty'].sum()
          .unstack(fill_value=0).reindex(columns=months, fill_value=0))

cat = pd.DataFrame({
    'avg_monthly_qty': dcat.mean(axis=1),
    'cv': dcat.std(axis=1) / dcat.mean(axis=1),
    'active_months': (dcat > 0).sum(axis=1),
    'revenue': d.groupby('category')['sales'].sum()
}).sort_values('revenue', ascending=False)

share = cat['revenue'] / cat['revenue'].sum()
prior = share.cumsum() - share
cat['ABC'] = np.where(prior < 0.80, 'A', np.where(prior < 0.95, 'B', 'C'))
cat['XYZ'] = np.where(cat['cv'] <= 0.5, 'X', np.where(cat['cv'] <= 1.0, 'Y', 'Z'))

eligible = cat[cat['active_months'] >= 28]
print(len(eligible), "eligible categories;", len(cat) - len(eligible), "set aside (short history)")
print(pd.crosstab(eligible['ABC'], eligible['XYZ'], margins=True))
eligible.round(2).to_csv('../data/category_segments.csv')
eligible.round(2)

24 eligible categories; 8 set aside (short history)
XYZ   X  Y  All
ABC            
A     7  0    7
B     3  0    3
C    10  4   14
All  20  4   24


,avg_monthly_qty,cv,active_months,revenue,ABC,XYZ
category,,,,,,
Fishing,502.42,0.05,33,6631668.58,A,X
Cleats,2136.09,0.05,33,4237395.21,A,X
Camping & Hiking,398.24,0.04,33,3942337.30,A,X
Cardio Equipment,1087.06,0.08,33,3526889.85,A,X
Women's Apparel,1822.85,0.06,33,3007700.00,A,X
Water Sports,450.15,0.06,33,2976851.53,A,X
Indoor/Outdoor Games,1674.70,0.06,33,2762144.67,A,X
Men's Footwear,643.42,0.05,33,2760077.79,B,X
Shop By Sport,948.61,0.07,33,1252395.88,B,X
